# Chunk Coverage Versus Joint Scoring

This notebook is a small visual workbench for one recurring confusion:

- if the whole document is chunked, doesn't that already mean the whole document is being used?

Short answer:

- yes, chunking can cover the whole document
- no, retrieval often still scores chunks rather than the whole document jointly

Use the visual first, then the executable toy example below.

![Chunk coverage versus joint scoring](../assets/chunk_coverage_vs_joint_scoring.svg)


In [ ]:
from pathlib import Path
import sys

import numpy as np


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "python" / "kayak").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository or one of its subdirectories.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(REPO_ROOT / "python"))

import kayak

print("Working from repo root:", REPO_ROOT)
print("Backends available here:", kayak.available_backends())


In [ ]:
DIM = 64
TOKEN_TO_INDEX: dict[str, int] = {}


def token_vector(token: str) -> np.ndarray:
    index = TOKEN_TO_INDEX.setdefault(token, len(TOKEN_TO_INDEX))
    if index >= DIM:
        raise ValueError("Increase DIM for this notebook example.")
    vector = np.zeros(DIM, dtype=np.float32)
    vector[index] = np.float32(1.0)
    return vector


def encode_tokens(tokens: list[str]) -> np.ndarray:
    return np.stack([token_vector(token) for token in tokens])


def mean_vec(tokens: list[str]) -> np.ndarray:
    return encode_tokens(tokens).mean(axis=0, dtype=np.float32, keepdims=True).astype(np.float32)


def flatten_document_chunks(documents_by_chunk: dict[str, list[list[str]]]) -> dict[str, list[str]]:
    flattened: dict[str, list[str]] = {}
    for doc_id, chunks in documents_by_chunk.items():
        tokens: list[str] = []
        for chunk_tokens in chunks:
            tokens.extend(chunk_tokens)
        flattened[doc_id] = tokens
    return flattened


def build_chunked_onevec_index(
    documents_by_chunk: dict[str, list[list[str]]],
) -> tuple[kayak.LateIndex, dict[str, str], dict[str, list[str]]]:
    chunk_ids: list[str] = []
    chunk_vectors: list[np.ndarray] = []
    parent_by_chunk: dict[str, str] = {}
    chunk_tokens_by_id: dict[str, list[str]] = {}
    for doc_id, chunks in documents_by_chunk.items():
        for chunk_index, chunk_tokens in enumerate(chunks):
            chunk_id = f"{doc_id}::chunk{chunk_index}"
            chunk_ids.append(chunk_id)
            chunk_vectors.append(mean_vec(chunk_tokens))
            parent_by_chunk[chunk_id] = doc_id
            chunk_tokens_by_id[chunk_id] = list(chunk_tokens)
    return kayak.documents(chunk_ids, chunk_vectors).pack(), parent_by_chunk, chunk_tokens_by_id


def dedup_parent_docs(chunk_hits, parent_by_chunk: dict[str, str], *, k: int) -> tuple[str, ...]:
    ranked_doc_ids: list[str] = []
    seen: set[str] = set()
    for hit in chunk_hits:
        doc_id = parent_by_chunk[hit.doc_id]
        if doc_id in seen:
            continue
        seen.add(doc_id)
        ranked_doc_ids.append(doc_id)
        if len(ranked_doc_ids) >= k:
            break
    return tuple(ranked_doc_ids)


def aggregate_parent_scores(
    chunk_hits,
    parent_by_chunk: dict[str, str],
    *,
    mode: str,
) -> list[tuple[str, float]]:
    scores_by_parent: dict[str, list[float]] = {}
    for hit in chunk_hits:
        scores_by_parent.setdefault(parent_by_chunk[hit.doc_id], []).append(float(hit.score))
    if mode == "max":
        parent_scores = [
            (doc_id, max(scores))
            for doc_id, scores in scores_by_parent.items()
        ]
    elif mode == "sum":
        parent_scores = [
            (doc_id, sum(scores))
            for doc_id, scores in scores_by_parent.items()
        ]
    else:
        raise ValueError(f"Unknown aggregation mode: {mode}")
    return sorted(parent_scores, key=lambda item: item[1], reverse=True)


## A Tiny Case Where Coverage Is Real But Joint Scoring Is Missing

Document A contains the full evidence, but it is split across three chunks.
Document B contains one sharp local match for only part of the query.

We compare:

- exact late interaction over the full document
- chunked one-vector retrieval over all chunks from all documents
- two parent-document aggregation rules over those chunk scores


In [ ]:
query_tokens = ["cancel", "subscription", "refund"]

documents_by_chunk = {
    "doc-a": [
        ["cancel", "noise1"],
        ["subscription", "noise2"],
        ["refund", "noise3"],
    ],
    "doc-b": [
        ["cancel", "cancel", "cancel", "cancel"],
    ],
    "doc-c": [
        ["invoice", "billing"],
        ["account", "payment"],
    ],
}

full_documents = flatten_document_chunks(documents_by_chunk)
exact_index = kayak.documents(
    list(full_documents),
    [encode_tokens(tokens) for tokens in full_documents.values()],
).pack()
chunk_index, parent_by_chunk, chunk_tokens_by_id = build_chunked_onevec_index(documents_by_chunk)

exact_hits = kayak.search(
    kayak.query(encode_tokens(query_tokens)),
    exact_index,
    k=3,
    backend=kayak.NUMPY_REFERENCE_BACKEND,
)
chunk_hits = kayak.search(
    kayak.query(mean_vec(query_tokens)),
    chunk_index,
    k=chunk_index.document_count,
    backend=kayak.NUMPY_REFERENCE_BACKEND,
)

print("Query tokens:", query_tokens)
print()
print("Full documents:")
for doc_id, tokens in full_documents.items():
    print(f"  {doc_id}: {tokens}")
print()
print("Exact full-document ranking:")
print([(hit.doc_id, round(float(hit.score), 4)) for hit in exact_hits])
print()
print("Global chunk ranking:")
for hit in chunk_hits:
    print(
        hit.doc_id,
        parent_by_chunk[hit.doc_id],
        round(float(hit.score), 4),
        chunk_tokens_by_id[hit.doc_id],
    )
print()
print("If I just keep the first chunk that makes each parent visible:")
print(dedup_parent_docs(chunk_hits, parent_by_chunk, k=3))


The exact path says `doc-a` wins because the whole document contains all three evidence tokens.

The chunked path says `doc-b::chunk0` wins because that one chunk is the strongest local match.
That is the distinction:

- coverage is real
- joint document scoring is still missing


In [ ]:
parent_max = aggregate_parent_scores(chunk_hits, parent_by_chunk, mode="max")
parent_sum = aggregate_parent_scores(chunk_hits, parent_by_chunk, mode="sum")
top_context_chunks = chunk_hits[:2]

print("If I aggregate parents by max chunk score:")
print([(doc_id, round(score, 4)) for doc_id, score in parent_max])
print()
print("If I aggregate parents by sum of all retrieved chunk scores:")
print([(doc_id, round(score, 4)) for doc_id, score in parent_sum])
print()
print("If I only send the top 2 chunks to the model, they are:")
for hit in top_context_chunks:
    print(
        hit.doc_id,
        parent_by_chunk[hit.doc_id],
        round(float(hit.score), 4),
        chunk_tokens_by_id[hit.doc_id],
    )


## What To Keep In Mind

- Chunking the whole document means the whole document is represented.
- It does **not** mean the whole document is jointly scored.
- Hybrid retrieval can improve chunk recall, but it still often retrieves chunks first.
- Parent grouping or aggregation is an extra design choice.
- Exact late interaction is useful as a reference path because it keeps the full document as one scored object.
